In [24]:
import sys
import importlib.util
from langchain_google_genai import ChatGoogleGenerativeAI
import google.auth
from google.cloud import bigquery

In [25]:

PROJECT_ID = "southwell-infra"
TABLE_ID = "southwell-infra.Odibets_Raw.odibets_transaction_new"
TIMEZONE = "Africa/Nairobi"

# Discover the Google credentials available to this notebook.
credentials, detected_project = google.auth.default()

# Create the client: our Python connection to BigQuery.
bq_client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials,
)

# # Check that we can run queries without scanning transaction data.
# rows = bq_client.query("SELECT 1 AS connection_ok").result()
# print("Query connection:", next(iter(rows)).connection_ok == 1)

# # Check access to the table's metadata.
# table = bq_client.get_table(TABLE_ID)
# print("Table accessible:", table.full_table_id)

#### PROMPT

In [26]:
SYSTEM_PROMPT = """
You are Odi Kenya's data analysis assistant.

Purpose:
- Answer questions using BigQuery transaction data.
- Export analytical results to Google Sheets when requested.
- Return the actual Sheet URL with concise findings.

Available business table:
southwell-infra.Odibets_Raw.odibets_transaction_new

Business context:
This table combines sportsbook, casino, virtual-game, and wallet
transactions, including bets, settlements, deposits, withdrawals,
reversals, and bonuses.

Columns:
- transaction_id INT64: identifies a transaction.
  The raw table can contain multiple versions of a transaction.
- profile_id INT64: identifies a bettor.
- reference_id INT64: identifier from the originating system.
  Related transactions can share it; it is not unique.
  Do not assume it alone safely links transactions across providers.
- reference_type_id INT64: reference classification; meaning unconfirmed.
- created_by STRING and channel_id STRING: together classify transaction
  source and type. Use mappings supplied by the user; otherwise establish
  the relevant mapping before interpreting a label.
- amount STRING: transaction amount. Use SAFE_CAST(amount AS NUMERIC).
  Follow user-supplied sign conventions. Report failed conversions
  rather than silently treating them as zero or complete totals.
- running_balance NUMERIC: wallet balance after the transaction.
- created DATETIME: transaction creation time; its timezone convention
  remains unconfirmed unless specified by the user.
- modified TIMESTAMP: transaction modification time.

Technical context:
- _PARTITIONTIME is ingestion partition time, not transaction time.
- datastream_metadata contains streaming information, not business metrics.

Confirmed deduplication:
- Deduplicate by transaction_id, keeping the latest modified row.
- Use:
  QUALIFY ROW_NUMBER() OVER (
      PARTITION BY transaction_id
      ORDER BY modified DESC
  ) = 1
- Deduplicate in a CTE or subquery before transaction-type filters
  and aggregations.
- Never deduplicate by reference_id.
- Equal modified timestamps do not uniquely order conflicting versions.
  Do not invent a tie-breaking rule.
- Ensure filters do not unintentionally exclude versions needed
  to identify the latest transaction state.

Query efficiency:
- Every source-table query must include bounded _PARTITIONTIME
  conditions before deduplication.
- Group the partition conditions with OR _PARTITIONTIME IS NULL
  inside parentheses before combining them with other filters.
- Keep a created filter when the question concerns transaction time.
- _PARTITIONTIME uses UTC. Convert Nairobi boundaries appropriately
  and align ingestion bounds with the table's partition granularity.
- An ingestion window does not guarantee complete historical state.
  Account for known late arrivals; disclose unresolved coverage limits.
- Select only necessary columns.
- Never bypass or increase the query tool's billing allowance.
- If a query is rejected for cost, optimize within the requested period.
  Do not substitute an arbitrary reporting period.

Execution discipline:
- Treat labels, column mappings, amount conventions, and reporting periods
  explicitly supplied by the user as instructions for that request.
- Do not run exploratory queries merely to reconfirm supplied definitions.
- If the reporting period is missing, ask for it before querying.
- If another essential definition is missing, ask a focused question
  or perform a targeted discovery query.
- Interpret label fallback instructions as value matching, not COALESCE.
  COALESCE handles NULL values; it does not search for a label.
- Use date expressions directly in analytical SQL; do not query
  separately just to determine the current date.
- Prefer a direct analytical query over multiple exploratory queries.
- Query only the allowed business table, not INFORMATION_SCHEMA.
- After two consecutive failed query attempts, stop and explain
  the blocker.

Connected tools:
- query_bigquery executes SQL and returns a result_id and preview.
- export_to_sheets exports the full saved result using its result_id.
- Use query_bigquery for data questions rather than merely suggesting SQL.
- Preview rows may be incomplete; do not treat them as the full result.
- When export is requested, call export_to_sheets after obtaining
  the final analytical result, using its exact result_id.
- Do not export exploratory checks unless requested.
- Claim export success only when the export tool returns a URL.
- Include that URL in the final answer.
- If export fails, report the failure without inventing a link.

Response rules:
- Use Africa/Nairobi to interpret relative dates.
- Never invent labels, query results, or Sheet URLs.
- Distinguish observations from possible explanations.
- Do not conclude fraud solely because activity is unusual.
- State material assumptions and data-quality limitations briefly.
Output style:
- Present the requested findings directly, followed by the Sheet link.
- Use a compact table when appropriate.
- Do not routinely explain SQL, casting, deduplication, or tool execution unless asked
- Mention only material limitations that affect interpretation.
- Provide technical methodology only when requested.
-When asked to give the SQL query used, do provide please. but only when asked

Output and export rules:
- Apply these rules to every result, whether raw or aggregated.
- If a result contains more than 3 rows, export the full result using
  export_to_sheets. Do not reproduce its rows in the response.
- For these larger results, respond only with the row count, a brief
  description, and the actual Sheet link.
- If a result contains 3 rows or fewer, display it concisely.
- If the user explicitly requests an export, export regardless of row count.
- Provide previews, detailed analysis, or SQL only when requested.
- Never invent a Sheet URL or claim export succeeded if the tool failed.
"""

AGENT_PROMPT = SYSTEM_PROMPT

In [28]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# PROJECT_ID = "southwell-infra"
# LOCATION = "global"
# MODEL_ID = "gemini-3.8-flash"
# # LOCATION = "europe-west1"
# # MODEL_ID = "claude-sonnet-4-6"

# brain = ChatGoogleGenerativeAI(
#     model=MODEL_ID,
#     vertexai=True,
#     project=PROJECT_ID,
#     location=LOCATION,
#     max_retries=2,
# )


# def show_text(message):
#     if isinstance(message.content, str):
#         print(message.content)
#     else:
#         print("\n".join(
#             block["text"]
#             for block in message.content
#             if isinstance(block, dict) and block.get("type") == "text"
#         ))

PROJECT_ID = "southwell-infra"

# Change these together when selecting another provider.
MODEL_PROVIDER = "gemini_vertex"
MODEL_ID = "gemini-3.8-flash"
LOCATION = "global"
# MODEL_PROVIDER = "anthropic_vertex"
# MODEL_ID = "claude-sonnet-4-6"
# LOCATION = "europe-west1"


def create_brain():
    """Create the connector for the selected model provider."""

    if MODEL_PROVIDER == "gemini_vertex":
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model=MODEL_ID,
            vertexai=True,
            project=PROJECT_ID,
            location=LOCATION,
            max_retries=2,
        )

    if MODEL_PROVIDER == "anthropic_vertex":
        from langchain_google_vertexai.model_garden import (
            ChatAnthropicVertex,
        )

        return ChatAnthropicVertex(
            model_name=MODEL_ID,
            project=PROJECT_ID,
            location=LOCATION,
            max_tokens=4096,
            max_retries=2,
        )

    if MODEL_PROVIDER == "openai":
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(
            model=MODEL_ID,
            max_retries=2,
        )

    if MODEL_PROVIDER == "anthropic":
        from langchain_anthropic import ChatAnthropic

        return ChatAnthropic(
            model=MODEL_ID,
            max_tokens=4096,
            max_retries=2,
        )

    raise ValueError(f"Unsupported provider: {MODEL_PROVIDER!r}")


def show_text(message):
    if isinstance(message.content, str):
        print(message.content)
    else:
        print("\n".join(
            block["text"]
            for block in message.content
            if isinstance(block, dict) and block.get("type") == "text"
        ))


brain = create_brain()

print("Provider:", MODEL_PROVIDER)
print("Connector:", type(brain).__name__)

# Test the model connection without querying BigQuery.
response = brain.invoke("Reply with: Model connection successful.")
show_text(response)

Provider: gemini_vertex
Connector: ChatGoogleGenerativeAI
Model connection successful.


In [29]:
import json
from uuid import uuid4

from google.cloud import bigquery
from langchain.tools import tool

bq_client = bigquery.Client(project=PROJECT_ID)

TABLE_ID = "southwell-infra.Odibets_Raw.odibets_transaction_new"
MAX_BYTES_BILLED = 1_000_000_000  # 1 GB per query
MAX_RESULT_ROWS = 10_000

# Keep results in Python for later export to Sheets.
query_results = {}


@tool
def query_bigquery(sql: str) -> dict:
    """Execute a read-only BigQuery SELECT query on the allowed transaction table.

    Use bounded ingestion partition filters and the agreed deduplication rule.
    Return aggregated results where possible. Returns a result ID, row count,
    and up to 30 preview rows. Full results remain available for Sheets export.
    """
    try:
        # Validate SQL and estimate its scan size without executing it.
        estimate = bq_client.query(
            sql,
            job_config=bigquery.QueryJobConfig(
                dry_run=True,
                use_query_cache=False,
            ),
        )

        if estimate.statement_type != "SELECT":
            return {"error": "Only a single SELECT query is allowed."}

        referenced_tables = {
            f"{t.project}.{t.dataset_id}.{t.table_id}"
            for t in (estimate.referenced_tables or [])
        }

        if referenced_tables != {TABLE_ID}:
            return {"error": f"Query must read only {TABLE_ID}."}

        estimated_bytes = estimate.total_bytes_processed

        if estimated_bytes is None:
            return {"error": "Could not estimate query size; execution blocked."}

        if estimated_bytes > MAX_BYTES_BILLED:
            return {
                "error": "Query exceeds the 1 GB scan allowance. Narrow the query.",
                "estimated_gb": round(estimated_bytes / 30_000_000_000  , 3),
            }

        # The execution cap remains in place even after the dry run.
        job = bq_client.query(
            sql,
            job_config=bigquery.QueryJobConfig(
                maximum_bytes_billed=MAX_BYTES_BILLED,
            ),
        )
        rows = job.result()

        if rows.total_rows > MAX_RESULT_ROWS:
            return {
                "error": "Result exceeds 10,000 rows. Aggregate or narrow the query.",
                "row_count": rows.total_rows,
                "job_id": job.job_id,
            }

        df = rows.to_dataframe(create_bqstorage_client=False)
        result_id = uuid4().hex

        query_results[result_id] = {
            "data": df,
            "sql": sql,
            "job_id": job.job_id,
        }

        return {
            "result_id": result_id,
            "row_count": len(df),
            "preview_rows": json.loads(
                df.head(30).to_json(orient="records", date_format="iso")
            ),
            "preview_is_partial": len(df) > 30,
            "bytes_processed": job.total_bytes_processed,
        }

    except Exception as exc:
        return {"error": str(exc)}

In [30]:
import google.auth
from google.auth import impersonated_credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

SHEETS_SERVICE_ACCOUNT = (
    "southwell-data-team-sc@southwell-infra.iam.gserviceaccount.com"
)

source_credentials, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

sheets_credentials = impersonated_credentials.Credentials(
    source_credentials=source_credentials,
    target_principal=SHEETS_SERVICE_ACCOUNT,
    target_scopes=["https://www.googleapis.com/auth/spreadsheets"],
    lifetime=3600,
)

# Request credentials now so authentication errors appear immediately.
sheets_credentials.refresh(Request())

sheets_client = build(
    "sheets",
    "v4",
    credentials=sheets_credentials,
    cache_discovery=False,
)

print("Sheets authentication ready:", SHEETS_SERVICE_ACCOUNT)

Sheets authentication ready: southwell-data-team-sc@southwell-infra.iam.gserviceaccount.com


In [31]:
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
from langchain.tools import tool

SPREADSHEET_ID = "19o8EBmTQSwhuPJd0L3rl3-ZttIwkAQ9cVtQBxAVQn2Q"


@tool
def export_to_sheets(result_id: str) -> dict:
    """Export a saved BigQuery result to a new tab and return its URL.

    Use the result_id returned by query_bigquery.
    """
    if result_id not in query_results:
        return {"error": "Unknown result_id. Run the BigQuery query first."}

    saved = query_results[result_id]
    df = saved["data"]

    # Reuse a successful export instead of creating duplicate tabs.
    if "sheet_url" in saved:
        return {
            "url": saved["sheet_url"],
            "row_count": len(df),
        }

    if len(df.columns) == 0:
        return {"error": "Result has no columns to export."}

    # Retain the tab name so interrupted exports can be retried.
    if "sheet_tab_name" not in saved:
        timestamp = datetime.now(
            ZoneInfo("Africa/Nairobi")
        ).strftime("%Y%m%d_%H%M%S")

        saved["sheet_tab_name"] = (
            f"Analysis_{timestamp}_{result_id[:6]}"
        )

    title = saved["sheet_tab_name"]

    def cell_value(value):
        if pd.isna(value):
            return ""
        if isinstance(value, (str, bool, int, float)):
            return value
        return str(value)

    values = [[str(column) for column in df.columns]]
    values += [
        [cell_value(value) for value in row]
        for row in df.itertuples(index=False, name=None)
    ]

    try:
        workbook = sheets_client.spreadsheets().get(
            spreadsheetId=SPREADSHEET_ID,
            fields="sheets.properties",
        ).execute()

        properties = next(
            (
                sheet["properties"]
                for sheet in workbook.get("sheets", [])
                if sheet["properties"]["title"] == title
            ),
            None,
        )

        if properties is None:
            created = sheets_client.spreadsheets().batchUpdate(
                spreadsheetId=SPREADSHEET_ID,
                body={
                    "requests": [{
                        "addSheet": {
                            "properties": {
                                "title": title,
                                "gridProperties": {
                                    "rowCount": max(len(values), 100),
                                    "columnCount": max(len(df.columns), 1),
                                    "frozenRowCount": 1,
                                },
                            }
                        }
                    }]
                },
            ).execute()

            properties = (
                created["replies"][0]["addSheet"]["properties"]
            )

        sheet_id = properties["sheetId"]

        # Write headers and data in batches.
        for start in range(0, len(values), 500):
            sheets_client.spreadsheets().values().update(
                spreadsheetId=SPREADSHEET_ID,
                range=f"'{title}'!A{start + 1}",
                valueInputOption="RAW",
                body={"values": values[start:start + 500]},
            ).execute()

        url = (
            f"https://docs.google.com/spreadsheets/d/"
            f"{SPREADSHEET_ID}/edit#gid={sheet_id}"
        )

        saved["sheet_url"] = url

        return {
            "url": url,
            "row_count": len(df),
            "tab_name": title,
        }

    except Exception as exc:
        return {
            "error": str(exc),
            "tab_name": title,
            "note": (
                "Export may be incomplete. "
                "Retry with the same result_id."
            ),
        }

In [32]:
from uuid import uuid4
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

agent = create_agent(
    model=brain,
    tools=[query_bigquery, export_to_sheets],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memory,
)

conversation_config = {
    "configurable": {"thread_id": uuid4().hex},
    "recursion_limit": 12,
}

print("Conversation:", conversation_config["configurable"]["thread_id"])


# def ask_agent(question: str):
#     result = agent.invoke(
#         {"messages": [{"role": "user", "content": question}]},
#         config=conversation_config,
#     )

#     show_text(result["messages"][-1])
#     return result

Conversation: 1e9c1c30c614492890c7d97eb3cf101c


In [33]:

# from langgraph.errors import GraphRecursionError

# request = {
#     "messages": [{
#         "role": "user",
#         "content": ("Now show the top 5 instead, keeping the same reporting period "
# "and definitions. Export the results and return the Sheet link."
#         ),
#     }]
# }

# try:
#     result = agent.invoke(
#         request,
#         config=conversation_config,
#     )

#     show_text(result["messages"][-1])

# except GraphRecursionError:
#     print("The agent reached its execution-step limit before finishing.")

def send_message(text):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": text}]},
        config=conversation_config,
    )
    show_text(result["messages"][-1])


send_message("Ohh and we arent yet at 1 am. so do from 12 AM.TOP 200"
             
)
print("\n--- Memory check ---")



state = agent.get_state(conversation_config)
print("\nSaved messages:", len(state.values.get("messages", [])))

Could you please clarify what you would like the **Top 200** ranked by, and for which date? 

For example:
1. **Target metric/entity:** Top 200 players by deposits, turnover/stakes, net wins, withdrawals, or top individual transactions?
2. **Date:** Is this for today (from 12:00 AM midnight to the current time, Africa/Nairobi time)?

--- Memory check ---

Saved messages: 2


In [34]:
# # RESULT VERIFICATION — does not execute queries

# for message in reversed(result["messages"]):
#     if message.type != "tool" or message.name != "query_bigquery":
#         continue

#     tool_output = json.loads(message.content)
#     result_id = tool_output.get("result_id")

#     if result_id and result_id in query_results:
#         saved = query_results[result_id]

#         print("Result ID:", result_id)
#         print("BigQuery job ID:", saved["job_id"])
#         print("Sheet URL:", saved.get("sheet_url", "Not exported"))

#         print("\nACTUAL EXECUTED SQL:")
#         print(saved["sql"])

#         print("\nSAVED RESULTS:")
#         display(saved["data"])
#         break
# else:
#     print("No successful BigQuery result found in this conversation.")